In [1]:
files = [
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_NC.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_NC.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_NC.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_NC.csv",
]

In [2]:
import pandas as pd
for file in files:
    df = pd.read_csv(file)
    todos_ok = df["SubjectID"].str.match(r".*_\d{2}$").all()

    print(todos_ok)


True
True
True
True
True
True
True
True
True
True
True
True


In [3]:
import pandas as pd
import json
import os
import re

resultado = {}

todas_modalidades = set()
todas_regiones = set()
num_features = None

# --- PRIMER PASO: leer todo y guardar info ---
data_tmp = []

for file in files:
    df = pd.read_csv(file)

    filename = os.path.basename(file)
    match = re.search(r'automaticsegm_(\w+)_(\w+)\.csv', filename)
    if not match:
        continue

    modalidad = match.group(1)
    region = match.group(2)

    todas_modalidades.add(modalidad)
    todas_regiones.add(region)

    df["SubjectID_clean"] = df["SubjectID"].str.replace(r"_\d{2}$", "", regex=True)

    feature_cols = [c for c in df.columns if c not in ["SubjectID", "SubjectID_clean"]]

    if num_features is None:
        num_features = len(feature_cols)

    data_tmp.append((df, modalidad, region, feature_cols))


# --- SEGUNDO PASO: construir diccionario ---
for df, modalidad, region, feature_cols in data_tmp:
    for _, row in df.iterrows():
        sid = row["SubjectID_clean"]

        if sid not in resultado:
            resultado[sid] = {}

        if modalidad not in resultado[sid]:
            resultado[sid][modalidad] = {}

        resultado[sid][modalidad][region] = row[feature_cols].tolist()


# --- TERCER PASO: rellenar faltantes con None ---
for sid in resultado:
    for modalidad in todas_modalidades:
        if modalidad not in resultado[sid]:
            resultado[sid][modalidad] = {}

        for region in todas_regiones:
            if region not in resultado[sid][modalidad]:
                resultado[sid][modalidad][region] = [None] * num_features


# --- guardar json ---
with open("../data/radiomic_not_processed.json", "w") as f:
    json.dump(resultado, f, indent=2)

/tmp/ipykernel_118887/4216731768.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["SubjectID_clean"] = df["SubjectID"].str.replace(r"_\d{2}$", "", regex=True)


In [4]:
import json

with open("../data/radiomic_not_processed.json", "r") as f:
    data = json.load(f)

In [5]:
import json
from collections import defaultdict, Counter

with open("../data/radiomic_not_processed.json", "r") as f:
    data = json.load(f)

num_subjects = len(data)

modalidades_counter = Counter()
regiones_counter = Counter()
feature_lengths = []
missing = []

for sid, contenido in data.items():
    if not isinstance(contenido, dict):
        missing.append(sid)
        continue

    for modalidad, regiones in contenido.items():
        modalidades_counter[modalidad] += 1

        if not isinstance(regiones, dict):
            missing.append((sid, modalidad))
            continue

        for region, features in regiones.items():
            regiones_counter[region] += 1

            if isinstance(features, list):
                feature_lengths.append(len(features))
            else:
                missing.append((sid, modalidad, region))

# --- imprimir resumen ---
print("📊 RESUMEN GENERAL")
print(f"Número de sujetos: {num_subjects}")

print("\n🧠 Modalidades:")
for k, v in modalidades_counter.items():
    print(f"  {k}: {v}")

print("\n🧩 Regiones:")
for k, v in regiones_counter.items():
    print(f"  {k}: {v}")

print("\n📏 Features:")
if feature_lengths:
    print(f"  Min: {min(feature_lengths)}")
    print(f"  Max: {max(feature_lengths)}")
    print(f"  Media: {sum(feature_lengths)/len(feature_lengths):.2f}")
    print(f"  Valores únicos: {set(feature_lengths)}")

print("\n⚠️ Problemas detectados:")
print(f"  Entradas raras: {len(missing)}")
if missing[:5]:
    print("  Ejemplos:", missing[:5])

📊 RESUMEN GENERAL
Número de sujetos: 611

🧠 Modalidades:
  FLAIR: 611
  T1: 611
  T1GD: 611
  T2: 611

🧩 Regiones:
  ED: 2444
  ET: 2444
  NC: 2444

📏 Features:
  Min: 144
  Max: 144
  Media: 144.00
  Valores únicos: {144}

⚠️ Problemas detectados:
  Entradas raras: 0


In [6]:
import json
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed
import os

# ─────────────────────────────────────────────────────────────────────────────
# GlobalRadiomicsNormalizer
# ─────────────────────────────────────────────────────────────────────────────
class GlobalRadiomicsNormalizer:
    """
    Normalización por (modalidad, región) usando RobustScaler global sobre train.

    Estructura de entrada esperada:
        data[subject_id][modalidad][region] = List[float] | List[None]

    - Listas de None se ignoran en el fit y se devuelven como None en transform.
    - Un scaler independiente por cada par (modalidad, región).
    - Sin log1p: los features de radiomics ya tienen escalas comparables.
    """

    def __init__(self):
        self.scalers: dict[tuple, RobustScaler] = {}
        self.fitted = False

    def _collect_vectors(self, data: dict, modalidad: str, region: str):
        """Recoge todos los vectores válidos de train para un par (mod, reg)."""
        vecs = []
        sids = []
        for sid, mod_dict in data.items():
            vec = mod_dict.get(modalidad, {}).get(region)
            if vec is None or any(v is None for v in vec):
                continue
            arr = np.array(vec, dtype=float)
            if not np.isfinite(arr).all():
                arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            vecs.append(arr)
            sids.append(sid)
        return np.array(vecs) if vecs else None

    def fit(self, train_data: dict):
        """
        Ajusta un RobustScaler por cada (modalidad, región) usando solo train.
        """
        # Detectar modalidades y regiones automáticamente
        modalidades = set()
        regiones = set()
        for sid, mod_dict in train_data.items():
            for mod, reg_dict in mod_dict.items():
                modalidades.add(mod)
                for reg in reg_dict:
                    regiones.add(reg)

        print(f"[Normalizer] Modalidades: {sorted(modalidades)}")
        print(f"[Normalizer] Regiones:    {sorted(regiones)}")

        fitted_count = 0
        skipped = []

        for mod in sorted(modalidades):
            for reg in sorted(regiones):
                X = self._collect_vectors(train_data, mod, reg)
                if X is None or X.shape[0] == 0:
                    skipped.append((mod, reg))
                    continue

                scaler = RobustScaler()
                scaler.fit(X)
                self.scalers[(mod, reg)] = scaler
                fitted_count += 1
                print(f"  ✓ ({mod:6s}, {reg:4s}) → {X.shape[0]} sujetos, "
                      f"{X.shape[1]} features")

        if skipped:
            print(f"  ⚠ Sin datos suficientes (saltados): {skipped}")

        self.fitted = True
        print(f"\n[Normalizer] Scalers ajustados: {fitted_count}")

    def transform_subject(self, mod_dict: dict) -> dict:
        """Transforma un sujeto individual. Devuelve su mod_dict normalizado."""
        assert self.fitted, "Llama a fit() primero."
        result = {}
        for mod, reg_dict in mod_dict.items():
            result[mod] = {}
            for reg, vec in reg_dict.items():
                key = (mod, reg)
                # Si la lista es None o contiene Nones, se devuelve tal cual
                if vec is None or any(v is None for v in vec):
                    result[mod][reg] = vec
                    continue
                if key not in self.scalers:
                    # Par no visto en train: devolver sin escalar
                    result[mod][reg] = vec
                    continue
                arr = np.array(vec, dtype=float)
                arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
                scaled = self.scalers[key].transform(arr.reshape(1, -1)).flatten()
                result[mod][reg] = scaled.tolist()
        return result

    def transform(self, data: dict, n_jobs: int = -1) -> dict:
        """Transforma todo el dataset en paralelo."""
        assert self.fitted

        def _worker(sid):
            return sid, self.transform_subject(data[sid])

        results = Parallel(n_jobs=n_jobs, backend="loky", verbose=1)(
            delayed(_worker)(sid) for sid in data
        )
        return dict(results)


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline principal
# ─────────────────────────────────────────────────────────────────────────────

# 1. Cargar datos crudos
with open("../data/radiomic_not_processed.json", "r") as f:
    raw_data = json.load(f)

subject_ids = list(raw_data.keys())
print(f"Sujetos totales: {len(subject_ids)}")

# 2. Cargar partición predefinida
with open("../data/partitions_ids.json", "r") as f:
    splits = json.load(f)

train_ids = splits["train"]
val_ids   = splits["val"]
test_ids  = splits["test"]

print(f"Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

train_data = {sid: raw_data[sid] for sid in train_ids if sid in raw_data}
val_data   = {sid: raw_data[sid] for sid in val_ids   if sid in raw_data}
test_data  = {sid: raw_data[sid] for sid in test_ids  if sid in raw_data}

# 3. Fit sobre train únicamente
normalizer = GlobalRadiomicsNormalizer()
normalizer.fit(train_data)

# 4. Transform
print("\nTransformando train..."); train_scaled = normalizer.transform(train_data)
print("Transformando val...");   val_scaled   = normalizer.transform(val_data)
print("Transformando test...");  test_scaled  = normalizer.transform(test_data)

# 5. Unir todo en un único JSON con split como metadata por sujeto
combined = {}
for sid, mod_dict in train_scaled.items():
    combined[sid] = {"split": "train", "data": mod_dict}
for sid, mod_dict in val_scaled.items():
    combined[sid] = {"split": "val",   "data": mod_dict}
for sid, mod_dict in test_scaled.items():
    combined[sid] = {"split": "test",  "data": mod_dict}


with open("../data/radiomic_processed.json", "w") as f:
    json.dump(combined, f, indent=2)

print(f"\n✅ Guardado → {len(combined)} sujetos totales")
print(f"   train: {sum(1 for v in combined.values() if v['split'] == 'train')}")
print(f"   val:   {sum(1 for v in combined.values() if v['split'] == 'val')}")
print(f"   test:  {sum(1 for v in combined.values() if v['split'] == 'test')}")

Sujetos totales: 611
Train: 482 | Val: 60 | Test: 61
[Normalizer] Modalidades: ['FLAIR', 'T1', 'T1GD', 'T2']
[Normalizer] Regiones:    ['ED', 'ET', 'NC']
  ✓ (FLAIR , ED  ) → 472 sujetos, 144 features
  ✓ (FLAIR , ET  ) → 466 sujetos, 144 features
  ✓ (FLAIR , NC  ) → 463 sujetos, 144 features
  ✓ (T1    , ED  ) → 472 sujetos, 144 features
  ✓ (T1    , ET  ) → 466 sujetos, 144 features
  ✓ (T1    , NC  ) → 463 sujetos, 144 features
  ✓ (T1GD  , ED  ) → 472 sujetos, 144 features
  ✓ (T1GD  , ET  ) → 466 sujetos, 144 features
  ✓ (T1GD  , NC  ) → 463 sujetos, 144 features
  ✓ (T2    , ED  ) → 472 sujetos, 144 features
  ✓ (T2    , ET  ) → 466 sujetos, 144 features
  ✓ (T2    , NC  ) → 463 sujetos, 144 features

[Normalizer] Scalers ajustados: 12

Transformando train...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.6s
[Parallel(n_jobs=-1)]: Done 152 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 402 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done 472 out of 472 | elapsed:    5.8s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done  55 out of  55 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.0s


Transformando val...
Transformando test...


[Parallel(n_jobs=-1)]: Done  58 out of  58 | elapsed:    0.1s finished



✅ Guardado → 585 sujetos totales
   train: 472
   val:   55
   test:  58
